# Simple segmentation model trained from scratch

This notebook pairs files from `Input Micrographs` and `Output Micrographs`, extracts noisy pseudo-masks from the yellow outlines, and trains a small convolutional network from random weights. It does **not** use pretrained Cellpose weights. Poor accuracy is expected because the outputs are annotated preview images rather than clean masks.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
from tools.simple_model import prepare_real_dataset, train_simple_model, DATASET_DIRECTORY
print('Project:', PROJECT_ROOT)

## 1. Pair files and create pseudo-masks
Pairs with different dimensions are excluded to avoid spatially incorrect labels.

In [ ]:
manifest = prepare_real_dataset()
print('Accepted:', len(manifest['records']))
print('Excluded:', len(manifest['excluded']))

In [ ]:
from PIL import Image
from IPython.display import display
sample = sorted((DATASET_DIRECTORY / 'train').glob('*_img.png'))[0]
display(Image.open(sample))
display(Image.open(sample.with_name(sample.name.replace('_img.png', '_mask.png'))))

## 2. Train from random weights
Five epochs run on CPU. The compact model is saved as `models/simple_scratch_model.pt`; `trained_model.txt` makes the ImageJ plugin use it automatically.

In [ ]:
model_path, history = train_simple_model(epochs=5)
print('Model:', model_path)
history

## 3. Test with ImageJ
Open any saved input micrograph in ImageJ and compile/run `Run_AI_Detection.java`. The plugin calls `predict.py`, which loads this scratch-trained `.pt` model.